<div style="background:#1C3257;color:#F7F3EB;padding:22px 26px;border-radius:10px;font-family:Calibri,Arial,sans-serif"><div style="color:#E08A6E;font-size:12px;letter-spacing:2px;font-weight:bold">MINERÍA DE DATOS · UNIDAD 2 — DEL TEXTO AL SIGNIFICADO · UPCh 2026A</div><div style="font-size:26px;font-weight:bold;margin-top:6px">Lab 2 — Su primer motor de búsqueda</div><div style="font-style:italic;color:#C9D4E4;margin-top:8px">TF-IDF + similitud coseno, implementados desde cero · sin librerías de NLP</div></div>

## Reglas de entrega

- **Repo:** suban este notebook (ya ejecutado, con sus salidas) a su repositorio de GitHub Classroom de la organización `upch-mineria-2026a`.
- **`AI_USAGE.md` obligatorio:** si usaron IA en cualquier parte, declárenlo en ese archivo (qué herramienta, para qué celda, qué les dio y qué cambiaron). No declararlo se considera falta de honestidad académica.
- **Defensa oral (eliminatoria):** se les preguntará por *cualquier* celda. "Lo generó la IA" o "lo dijo el profesor" no son justificaciones válidas. Si no pueden explicar su código, no hay calificación.
- **Penalizaciones por entrega tardía:** 25% (<24 h), 50% (<48 h), rechazado (>48 h).
- Las celdas marcadas con `# TODO` y las preguntas en **negritas** son lo que se evalúa. El resto es andamiaje que ya viene resuelto para que enfoquen su tiempo en lo que importa.


## Objetivo

Dado el corpus que preprocesaron en el **Lab 1**, responder consultas en lenguaje libre devolviendo un **ranking por relevancia**. La regla del curso aplica: los algoritmos núcleo (TF, IDF, TF-IDF y coseno) se implementan **desde cero**. Está prohibido `TfidfVectorizer` u otra caja negra para resolver el ejercicio (solo se permite, opcionalmente, al final para *verificar*).


## 0 · Cargar el corpus procesado del Lab 1

Debe estar `corpus_procesado.json` en la misma carpeta. Si no, vuelvan a correr el Lab 1.

In [1]:
import json, math

with open('corpus_procesado.json', encoding='utf-8') as fh:
    corpus = json.load(fh)

documentos = [d['tokens'] for d in corpus]          # lista de listas de terminos
print(f'{len(corpus)} documentos.  Ejemplo {corpus[0]["id"]}:', documentos[0][:8])

14 documentos.  Ejemplo d01: ['fuerte', 'lluvia', 'provocar', 'inundación', 'colonia', 'sur', 'tuxtla', 'gutierrez']


**Reutilicen su `preprocesar`.** Péguenla aquí *idéntica* a la del Lab 1. Es indispensable que la consulta pase por **exactamente el mismo pipeline** que el corpus (este es el error más común: vectorizar la consulta distinto y que nada coincida).

In [2]:
import re, unicodedata
import spacy
from nltk.stem import SnowballStemmer

try:
    nlp = spacy.load('es_core_news_sm')
except OSError:
    from spacy.cli import download; download('es_core_news_sm')
    nlp = spacy.load('es_core_news_sm')

stemmer = SnowballStemmer('spanish')
stopwords_es = nlp.Defaults.stop_words
CONSERVAR = {'no', 'sin', 'contra'}
MIS_STOPWORDS = set(stopwords_es) - CONSERVAR

def normalizar(texto, quitar_acentos=False):
    texto = texto.lower()
    texto = re.sub(r'<[^>]+>', '', texto)
    texto = re.sub(r'https?://\S+', '', texto)
    if quitar_acentos:
        texto = unicodedata.normalize('NFD', texto)
        texto = "".join([c for c in texto if unicodedata.category(c) != 'Mn'])
    texto = unicodedata.normalize('NFC', texto)
    texto = re.sub(r'\s+', ' ', texto).strip()
    return texto

def tokens_lemma(texto):
    norm = normalizar(texto, quitar_acentos=True)
    doc = nlp(norm)
    tokens = []
    for token in doc:
        lemma = token.lemma_
        if lemma not in MIS_STOPWORDS and not token.is_punct and not token.is_space:
            tokens.append(lemma)
    return tokens

def preprocesar(texto):
    return tokens_lemma(texto)


## 1 · Indexar — TF, IDF y TF-IDF desde cero

Implementen las tres funciones siguiendo las fórmulas de la clase. La estructura es la del slide de "15 líneas".
- `tf(doc)` → frecuencia **relativa**: `f(t,d) / |d|` (importancia local).
- `idf(corpus)` → `ln(N / df(t))` (rareza global). Cuenten **documentos**, no apariciones.
- `tfidf(doc, idf_)` → `tf(t,d) · idf(t)`.


In [3]:
def tf(doc):
    """doc: lista de terminos -> dict termino:tf (frecuencia relativa)."""
    if not doc:
        return {}
    counts = {}
    for t in doc:
        counts[t] = counts.get(t, 0) + 1
    n = len(doc)
    return {t: count / n for t, count in counts.items()}

def idf(corpus):
    """corpus: lista de docs (cada uno lista de terminos) -> dict termino:idf."""
    N = len(corpus)
    if N == 0:
        return {}
    
    df_counts = {}
    for doc in corpus:
        unique_terms = set(doc)
        for t in unique_terms:
            df_counts[t] = df_counts.get(t, 0) + 1
            
    idf_dict = {}
    for t, df_t in df_counts.items():
        idf_dict[t] = math.log(N / df_t)
    return idf_dict

def tfidf(doc, idf_):
    """-> dict termino:peso tf-idf para un documento."""
    doc_tf = tf(doc)
    weights = {}
    for t, tf_val in doc_tf.items():
        weights[t] = tf_val * idf_.get(t, 0.0)
    return weights


In [4]:
# Construir el indice: un vector tf-idf (dict) por documento
IDF = idf(documentos)
INDICE = [tfidf(doc, IDF) for doc in documentos]

# Sanity check: el termino mas pesado de d04 (sequia/maiz) deberia ser tematico
import operator
top = sorted(INDICE[3].items(), key=operator.itemgetter(1), reverse=True)[:5]
print('Terminos top de', corpus[3]['id'], '->', top)

Terminos top de d04 -> [('gravemente', 0.16494108310095365), ('cultivo', 0.16494108310095365), ('maiz', 0.16494108310095365), ('frijol', 0.16494108310095365), ('fronterizo', 0.16494108310095365)]


## 2 · Procesar la consulta

La consulta es texto libre. Pásenla por el **mismo** `preprocesar`, conviértanla en vector con el **mismo** `IDF` del corpus (no recalculen IDF con la consulta dentro).

In [5]:
def vectorizar_consulta(texto):
    """texto libre -> vector tf-idf (dict) usando el IDF del corpus."""
    tokens = preprocesar(texto)
    query_tf = tf(tokens)
    query_vector = {}
    for t, tf_val in query_tf.items():
        query_vector[t] = tf_val * IDF.get(t, 0.0)
    return query_vector

# print(vectorizar_consulta('sequia en los cultivos'))


## 3 · Ranquear — similitud coseno

Implementen el coseno entre dos vectores dispersos (dicts) y la función `buscar` que devuelve el **top-k**.
$$\cos(\vec{q},\vec{d}) = \frac{\vec{q}\cdot\vec{d}}{\lVert\vec{q}\rVert\,\lVert\vec{d}\rVert}$$

In [6]:
def coseno(v1, v2):
    """Similitud coseno entre dos vectores dispersos (dicts). Maneja el caso de norma 0."""
    dot_product = 0.0
    smaller, larger = (v1, v2) if len(v1) < len(v2) else (v2, v1)
    for t, val in smaller.items():
        if t in larger:
            dot_product += val * larger[t]
            
    norm_v1 = math.sqrt(sum(val ** 2 for val in v1.values()))
    norm_v2 = math.sqrt(sum(val ** 2 for val in v2.values()))
    
    if norm_v1 == 0.0 or norm_v2 == 0.0:
        return 0.0
        
    return dot_product / (norm_v1 * norm_v2)

def buscar(consulta, k=5):
    """Devuelve los k documentos mas relevantes como (id, titulo, score)."""
    q = vectorizar_consulta(consulta)
    results = []
    for idx, doc_vector in enumerate(INDICE):
        score = coseno(q, doc_vector)
        doc_id = corpus[idx]['id']
        doc_titulo = corpus[idx]['titulo']
        results.append((doc_id, doc_titulo, score))
        
    results.sort(key=lambda x: x[2], reverse=True)
    return results[:k]


In [7]:
# Prueba: deberia rankear alto d02 (sequia/maiz) y d04
for id_, titulo, score in buscar('sequia y cultivos de maiz'):
    print(f'{score:.3f}  {id_}  {titulo}')

0.431  d04  Sequia afecta cultivos de maiz
0.083  d02  Crisis hidrica golpea la region
0.000  d01  Lluvias provocan inundaciones en Tuxtla
0.000  d03  Cafe de Chiapas rompe record de exportacion
0.000  d05  Turismo crece en el Canon del Sumidero


## 4 · Rómpanlo

Entender dónde falla un modelo vale más que celebrarlo donde funciona. Empezamos con el caso de clase:


In [8]:
print('Consulta: "problemas de agua"')
for id_, titulo, score in buscar('problemas de agua'):
    print(f'{score:.3f}  {id_}  {titulo}')

# Observen: d13 (agua potable) sale bien, pero d02 (crisis HIDRICA) — claramente relevante —
# queda hundido o en 0. 'agua' e 'hidrico' son cadenas distintas: TF-IDF no sabe que son sinonimos.

Consulta: "problemas de agua"
0.476  d13  Restablecen servicio de agua potable
0.000  d01  Lluvias provocan inundaciones en Tuxtla
0.000  d02  Crisis hidrica golpea la region
0.000  d03  Cafe de Chiapas rompe record de exportacion
0.000  d04  Sequia afecta cultivos de maiz


**4.a** Encuentren **2 consultas propias** donde su buscador devuelva resultados malos. Para cada una: muestren la salida y **expliquen la causa** (sinonimia, polisemia, falta de contexto, preprocesamiento agresivo, etc.). Traigan estas dos consultas a la próxima clase: con ellas abriremos BM25.

In [9]:
print('Consulta fallida 1: "temblor en el mar"')
for id_, titulo, score in buscar('temblor en el mar'):
    print(f'{score:.3f}  {id_}  {titulo}')


Consulta fallida 1: "temblor en el mar"
0.000  d01  Lluvias provocan inundaciones en Tuxtla
0.000  d02  Crisis hidrica golpea la region
0.000  d03  Cafe de Chiapas rompe record de exportacion
0.000  d04  Sequia afecta cultivos de maiz
0.000  d05  Turismo crece en el Canon del Sumidero


_Causa de la falla 1:_
**Sinonimia (Brecha de Vocabulario):** El usuario usó los términos "temblor" y "mar", pero el documento correspondiente `d06` ("Sismo de magnitud 5.1 frente a las costas...") utiliza los sinónimos exactos "sismo" y "costas". Dado que el modelo TF-IDF clásico realiza una coincidencia léxica exacta (exact keyword matching), es incapaz de identificar relaciones semánticas entre sinónimos y asigna una relevancia de 0.0 a todos los documentos del corpus.


In [10]:
print('Consulta fallida 2: "computadoras rapidas para laboratorios"')
for id_, titulo, score in buscar('computadoras rapidas para laboratorios'):
    print(f'{score:.3f}  {id_}  {titulo}')


Consulta fallida 2: "computadoras rapidas para laboratorios"
0.375  d07  UPCh inaugura laboratorio de IA
0.000  d01  Lluvias provocan inundaciones en Tuxtla
0.000  d02  Crisis hidrica golpea la region
0.000  d03  Cafe de Chiapas rompe record de exportacion
0.000  d04  Sequia afecta cultivos de maiz


_Causa de la falla 2:_
**Falta de contexto y brecha de abstracción conceptual:** El usuario busca computadoras veloces en laboratorios, lo cual se relaciona directamente con el documento `d07` ("laboratorio de IA equipado con GPUs"). Sin embargo, el término "computadora" y "rapido" no aparecen explícitamente en el texto, y TF-IDF no cuenta con un espacio semántico continuo (embeddings) o tesauro que le permita inferir que "GPUs" (unidades de procesamiento gráfico) y "aprendizaje automatico" implican computación de alta velocidad en un laboratorio de IA. El score resultante es 0.0.


## 5 · (Opcional) Verificación contra scikit-learn

Solo para **comprobar** que su implementación desde cero es correcta — no sustituye al ejercicio ni cuenta como entrega. Si sus rankings coinciden a grandes rasgos con los de `TfidfVectorizer`, van bien.

In [11]:
# from sklearn.feature_extraction.text import TfidfVectorizer
# from sklearn.metrics.pairwise import cosine_similarity
# docs_txt = [' '.join(t) for t in documentos]
# vec = TfidfVectorizer()
# X = vec.fit_transform(docs_txt)
# q = vec.transform([' '.join(preprocesar('sequia y cultivos de maiz'))])
# sims = cosine_similarity(q, X)[0]
# print(sorted(zip(sims, [d['id'] for d in corpus]), reverse=True)[:5])

## Entregables del Lab 2

- [ ] `tf`, `idf`, `tfidf`, `coseno`, `vectorizar_consulta` y `buscar` implementadas desde cero.
- [ ] El índice construido y la prueba de `buscar` ejecutada con salida.
- [ ] La demostración de la falla "problemas de agua" vs. "crisis hídrica" ejecutada.
- [ ] **2 consultas fallidas propias** con su causa explicada por escrito.
- [ ] `AI_USAGE.md` actualizado si usaron IA.
